# Notebook 07 — R\_HAT\_content: Register-Controlled Harmfulness Direction

Notebook 06 showed that R\_HAT (extracted from refusal/compliance pairs) is
register-dominant (η² = 0.64) rather than content-dominant (η² = 0.08).

**This notebook extracts a new direction** from the *content contrast* while
controlling for register:

```
R_HAT_content = PCA₁( h(harmful_imperative) − h(neutral_imperative) )
```

Using the same 20 quadruplets, but taking the harmful−neutral difference
*within imperative register only*, we extract a direction that by construction
cannot capture register (both sides are imperative requests).

**Questions:**
1. Does R\_HAT\_content converge cross-model? (LOO-AUC, Spearman)
2. Is it content-dominant in the 2×2 ANOVA?
3. Is it distinct from original R\_HAT? (cosine similarity)
4. Does it predict JBB category severity?

In [ ]:
# ── CELL 0 — CONFIGURATION ────────────────────────────────────────────────────
import os, gc, warnings
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr, mannwhitneyu, wilcoxon
from scipy.stats import f as f_dist
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from datetime import datetime

warnings.filterwarnings("ignore")

MANIFEST_DIR = Path("./results/manifests")
OUT_DIR      = Path("./results/out_content_direction")
STIMULI_PATH = Path("./data/register_content_stimuli.csv")
JBB_PATH     = Path("./data/jbb_behaviors.csv")

OUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "distilgpt2":          ("distilgpt2",                               "base",     False),
    "qwen2-0.5b":          ("Qwen/Qwen2-0.5B",                         "base",     True),
    "gemma-3-270m":        ("google/gemma-3-270m",                     "base",     False),
    "tinyllama-1.1b":      ("TinyLlama/TinyLlama-1.1B-Chat-v1.0",    "chat",     False),
    "phi3-mini":           ("microsoft/Phi-3-mini-4k-instruct",        "instruct", False),
    "llama3-8b-base":      ("meta-llama/Meta-Llama-3.1-8B",           "base",     False),
    "llama3-8b-instruct":  ("meta-llama/Meta-Llama-3.1-8B-Instruct",  "instruct", False),
    "mistral-7b-base":     ("mistralai/Mistral-7B-v0.1",              "base",     False),
    "mistral-7b-instruct": ("mistralai/Mistral-7B-Instruct-v0.3",     "instruct", False),
}

MANIFEST_NAMES = {
    "distilgpt2":          "distilgpt2",
    "qwen2-0.5b":          "qwen2_0_5b",
    "gemma-3-270m":        "gemma3_270m",
    "tinyllama-1.1b":      "tinyllama_1_1b",
    "phi3-mini":           "phi3_mini",
    "llama3-8b-base":      "llama3.1_8b_base",
    "llama3-8b-instruct":  "llama3.1_8b_instruct",
    "mistral-7b-base":     "mistral_7b_base",
    "mistral-7b-instruct": "mistral_7b_instruct",
}

LAYERS_FRAC     = 0.5
MAX_TOKENS_ACTS = 40
DTYPE           = torch.bfloat16

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB free")

In [ ]:
# ── CELL 1 — EXTRACTION FUNCTIONS ─────────────────────────────────────────────

def get_layer_idx(model, frac: float) -> int:
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        n = len(model.model.layers)
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        n = len(model.transformer.h)
    else:
        n = 32
    return max(0, min(int(frac * n), n - 1))

def mean_pool_hidden(model, tokenizer, text: str, layer_idx: int) -> np.ndarray:
    device = next(model.parameters()).device
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       max_length=MAX_TOKENS_ACTS).to(device)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True, return_dict=True)
    h = out.hidden_states[layer_idx + 1][0]
    return h.mean(dim=0).float().cpu().numpy()

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

stim_df = pd.read_csv(STIMULI_PATH)
jbb_df  = pd.read_csv(JBB_PATH)
print(f"Stimuli: {len(stim_df)}, JBB behaviors: {len(jbb_df)}")

In [ ]:
# ── CELL 2 — MAIN LOOP ────────────────────────────────────────────────────────
# For each model:
#   1. Extract hidden states for all 80 stimuli
#   2. Compute content diffs WITHIN imperative register (20 pairs)
#   3. PCA → R_HAT_content
#   4. Polarity correction
#   5. Project all 80 stimuli → register/content ANOVA
#   6. Project 100 JBB behaviors → tensions
#   7. Compare with original R_HAT

RESULTS_FILE  = OUT_DIR / "content_direction_results.csv"
TENSIONS_FILE = OUT_DIR / "jbb_tensions_content.csv"
ANOVA_FILE    = OUT_DIR / "anova_content_direction.csv"

if RESULTS_FILE.exists():
    done_models = set(pd.read_csv(RESULTS_FILE)["model"].unique())
    print(f"Cached: {len(done_models)} models")
else:
    done_models = set()

meta_rows     = []  # one row per model: summary stats
tension_rows  = []  # one row per model × behavior
anova_rows    = []  # one row per model: 2×2 ANOVA on stimuli

# Load cached if available
if RESULTS_FILE.exists():
    meta_rows = pd.read_csv(RESULTS_FILE).to_dict("records")
if TENSIONS_FILE.exists():
    tension_rows = pd.read_csv(TENSIONS_FILE).to_dict("records")
if ANOVA_FILE.exists():
    anova_rows = pd.read_csv(ANOVA_FILE).to_dict("records")

for model_name, (hf_id, group, trust_rc) in MODELS.items():
    if model_name in done_models:
        print(f"[SKIP] {model_name}")
        continue

    print(f"\n{'='*55}")
    print(f"  {model_name}")

    # ── Load model ──
    tok = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=trust_rc)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    cfg = AutoConfig.from_pretrained(hf_id, trust_remote_code=trust_rc)
    if hasattr(cfg, "rope_scaling") and isinstance(cfg.rope_scaling, dict):
        if "type" not in cfg.rope_scaling and "phi" not in hf_id.lower():
            cfg.rope_scaling["type"] = "linear"

    model = AutoModelForCausalLM.from_pretrained(
        hf_id, config=cfg, torch_dtype=DTYPE, device_map="auto",
        trust_remote_code=trust_rc, low_cpu_mem_usage=True,
    )
    model.eval()
    layer_idx = get_layer_idx(model, LAYERS_FRAC)
    print(f"  Layer: {layer_idx}")

    # ── Step 1: Extract hidden states for all 80 stimuli ──
    stim_hiddens = {}
    for idx, row in stim_df.iterrows():
        key = (row['quadruplet_id'], row['register'], row['content'])
        stim_hiddens[key] = mean_pool_hidden(model, tok, row['stimulus'], layer_idx)
    print(f"  Extracted {len(stim_hiddens)} stimulus hidden states")

    # ── Step 2: Content diffs within imperative register ──
    content_diffs = []
    for qid in range(1, 21):
        h_harm = stim_hiddens[(qid, 'imperative_request', 'harmful')]
        h_neut = stim_hiddens[(qid, 'imperative_request', 'neutral')]
        content_diffs.append(h_harm - h_neut)

    A = np.array(content_diffs)  # (20, hidden_dim)

    # ── Step 3: PCA → R_HAT_content ──
    pca = PCA(n_components=1)
    pca.fit(A)
    rhat_content = pca.components_[0]
    var_exp = float(pca.explained_variance_ratio_[0])
    print(f"  R_HAT_content extracted. var_exp = {var_exp:.3f}")

    # ── Step 4: Polarity correction ──
    # Use mean harmful - mean neutral projection as anchor
    mean_harm_proj = np.mean([np.dot(stim_hiddens[(q, 'imperative_request', 'harmful')], rhat_content) for q in range(1,21)])
    mean_neut_proj = np.mean([np.dot(stim_hiddens[(q, 'imperative_request', 'neutral')], rhat_content) for q in range(1,21)])
    polarity = +1 if mean_harm_proj > mean_neut_proj else -1
    rhat_content_corr = rhat_content * polarity
    print(f"  Polarity: {polarity} (harm_proj={mean_harm_proj:.4f}, neut_proj={mean_neut_proj:.4f})")

    # ── Step 5: Project 80 stimuli → ANOVA ──
    stim_tensions = []
    for idx, row in stim_df.iterrows():
        key = (row['quadruplet_id'], row['register'], row['content'])
        t = float(np.dot(stim_hiddens[key], rhat_content_corr))
        stim_tensions.append(t)

    stim_t = np.array(stim_tensions)
    stim_t_norm = (stim_t - stim_t.mean()) / stim_t.std()

    # 2×2 ANOVA
    registers = stim_df['register'].values
    contents  = stim_df['content'].values

    t_harm = stim_t_norm[contents == 'harmful']
    t_neut = stim_t_norm[contents == 'neutral']
    t_imp  = stim_t_norm[registers == 'imperative_request']
    t_dec  = stim_t_norm[registers == 'declarative_factual']

    grand_mean = stim_t_norm.mean()
    n_cell = 20

    mean_harm = t_harm.mean()
    mean_neut = t_neut.mean()
    mean_imp  = t_imp.mean()
    mean_dec  = t_dec.mean()

    mean_ih = stim_t_norm[(registers=='imperative_request') & (contents=='harmful')].mean()
    mean_in = stim_t_norm[(registers=='imperative_request') & (contents=='neutral')].mean()
    mean_dh = stim_t_norm[(registers=='declarative_factual') & (contents=='harmful')].mean()
    mean_dn = stim_t_norm[(registers=='declarative_factual') & (contents=='neutral')].mean()

    SS_content  = 2 * n_cell * ((mean_harm - grand_mean)**2 + (mean_neut - grand_mean)**2)
    SS_register = 2 * n_cell * ((mean_imp - grand_mean)**2 + (mean_dec - grand_mean)**2)

    predicted = [
        grand_mean + (mean_imp - grand_mean) + (mean_harm - grand_mean),
        grand_mean + (mean_imp - grand_mean) + (mean_neut - grand_mean),
        grand_mean + (mean_dec - grand_mean) + (mean_harm - grand_mean),
        grand_mean + (mean_dec - grand_mean) + (mean_neut - grand_mean),
    ]
    SS_interact = n_cell * sum((c - p)**2 for c, p in zip([mean_ih, mean_in, mean_dh, mean_dn], predicted))
    SS_total = stim_t_norm.var() * (len(stim_t_norm) - 1)
    SS_error = SS_total - SS_content - SS_register - SS_interact

    eta2_content  = SS_content / (SS_content + SS_error) if (SS_content + SS_error) > 0 else 0
    eta2_register = SS_register / (SS_register + SS_error) if (SS_register + SS_error) > 0 else 0
    eta2_interact = SS_interact / (SS_interact + SS_error) if (SS_interact + SS_error) > 0 else 0

    d_content = (t_harm.mean() - t_neut.mean()) / np.sqrt((t_harm.var() + t_neut.var()) / 2)
    d_register = (t_imp.mean() - t_dec.mean()) / np.sqrt((t_imp.var() + t_dec.var()) / 2)

    df_err = 76
    MS_err = SS_error / df_err if df_err > 0 else 1e-12
    F_c = (SS_content / 1) / MS_err
    F_r = (SS_register / 1) / MS_err
    p_c = 1 - f_dist.cdf(F_c, 1, df_err)
    p_r = 1 - f_dist.cdf(F_r, 1, df_err)

    print(f"  ANOVA: η²_content={eta2_content:.3f}  η²_register={eta2_register:.3f}  "
          f"η²_interact={eta2_interact:.3f}")
    print(f"  Cohen d: content={d_content:+.3f}  register={d_register:+.3f}")

    anova_rows.append({
        "model": model_name, "group": group,
        "eta2_content": eta2_content, "eta2_register": eta2_register,
        "eta2_interact": eta2_interact,
        "d_content": d_content, "d_register": d_register,
        "F_content": F_c, "F_register": F_r,
        "p_content": p_c, "p_register": p_r,
        "mean_ih": mean_ih, "mean_in": mean_in,
        "mean_dh": mean_dh, "mean_dn": mean_dn,
    })

    # ── Step 6: Project JBB behaviors ──
    for idx, row in jbb_df.iterrows():
        h = mean_pool_hidden(model, tok, row['behavior'], layer_idx)
        t = float(np.dot(h, rhat_content_corr))
        tension_rows.append({
            "model": model_name, "group": group,
            "track": row['track'], "category": row['category'],
            "behavior": row['behavior'], "tension": t,
        })
    print(f"  Projected 100 JBB behaviors")

    # ── Step 7: Compare with original R_HAT ──
    manifest_name = MANIFEST_NAMES[model_name]
    rhat_orig = np.load(MANIFEST_DIR / f"{manifest_name}.npy").astype(np.float32)
    df_orig = pd.read_csv(MANIFEST_DIR / f"{manifest_name}.csv")
    pol_orig = df_orig['polarity'].iloc[0]
    rhat_orig_corr = rhat_orig * pol_orig

    cos_with_orig = cosine_sim(rhat_content_corr, rhat_orig_corr)
    print(f"  cos(R_HAT_content, R_HAT_original) = {cos_with_orig:+.3f}")

    # Save R_HAT_content direction
    np.save(OUT_DIR / f"{manifest_name}_content.npy", rhat_content_corr)

    meta_rows.append({
        "model": model_name, "group": group,
        "var_exp": var_exp, "polarity": polarity,
        "cos_with_orig": cos_with_orig,
    })

    # ── Save checkpoints ──
    pd.DataFrame(meta_rows).to_csv(RESULTS_FILE, index=False)
    pd.DataFrame(tension_rows).to_csv(TENSIONS_FILE, index=False)
    pd.DataFrame(anova_rows).to_csv(ANOVA_FILE, index=False)

    del model, tok, stim_hiddens
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n✓ Done. Results in {OUT_DIR}")

In [ ]:
# ── CELL 3 — CROSS-MODEL CONVERGENCE ──────────────────────────────────────────

tensions_df = pd.read_csv(OUT_DIR / "jbb_tensions_content.csv")
meta_df     = pd.read_csv(OUT_DIR / "content_direction_results.csv")

print("="*70)
print("R_HAT_CONTENT — CROSS-MODEL CONVERGENCE")
print("="*70)

# Build tension matrix
models = sorted(tensions_df['model'].unique())
n_models = len(models)
n_behaviors = 100

T = np.zeros((n_models, n_behaviors))
for i, m in enumerate(models):
    sub = tensions_df[tensions_df['model']==m].sort_values('track')
    t = sub['tension'].values
    T[i] = (t - t.mean()) / t.std()

# Cross-model Spearman matrix
sp = np.zeros((n_models, n_models))
for i in range(n_models):
    for j in range(n_models):
        sp[i,j] = spearmanr(T[i], T[j])[0]

print("\nSpearman cross-model matrix:")
print(f"{'':>22}", end="")
for m in models:
    print(f"{m[:8]:>10}", end="")
print()
for i, m in enumerate(models):
    print(f"  {m:<20}", end="")
    for j in range(n_models):
        print(f"{sp[i,j]:>+10.3f}", end="")
    print()

# LOO-AUC
print("\n" + "="*70)
print("LOO-AUC (R_HAT_CONTENT)")
print("="*70)

for k in [10, 25]:
    aucs = []
    for i, target in enumerate(models):
        others_idx = [j for j in range(n_models) if j != i]
        consensus = T[others_idx].mean(axis=0)
        sorted_idx = np.argsort(consensus)[::-1]
        topk = list(sorted_idx[:k])
        botk = list(sorted_idx[-k:])
        labels = np.array([1]*k + [0]*k)
        scores = np.array([T[i][j] for j in topk + botk])
        aucs.append(roc_auc_score(labels, scores))

    print(f"\n  LOO-AUC (k={k}): mean={np.mean(aucs):.3f}  min={min(aucs):.3f}  max={max(aucs):.3f}")
    for m, auc in zip(models, aucs):
        print(f"    {m:<22} {auc:.3f}")

# Mean off-diagonal Spearman
offdiag = [sp[i,j] for i in range(n_models) for j in range(n_models) if i != j]
print(f"\nMean off-diagonal Spearman: {np.mean(offdiag):.3f} (range {min(offdiag):.3f}–{max(offdiag):.3f})")

In [ ]:
# ── CELL 4 — ANOVA COMPARISON: R_HAT_ORIG vs R_HAT_CONTENT ────────────────────

anova_content = pd.read_csv(OUT_DIR / "anova_content_direction.csv")

# Load original ANOVA from notebook 06
anova_orig = pd.read_csv("./results/out_register/anova_results.csv")

print("="*70)
print("ANOVA COMPARISON — R_HAT_ORIGINAL vs R_HAT_CONTENT")
print("="*70)

print(f"\n{'':>22} {'── R_HAT_ORIGINAL ──':>30} {'── R_HAT_CONTENT ──':>30}")
print(f"{'Model':<22} {'η²_cont':>8} {'η²_reg':>8} {'flag':>8} │ {'η²_cont':>8} {'η²_reg':>8} {'flag':>8}")
print("-"*82)

for _, (ro, rc) in enumerate(zip(
    anova_orig.sort_values('model').itertuples(),
    anova_content.sort_values('model').itertuples()
)):
    flag_o = "REG" if ro.eta2_register > ro.eta2_content * 2 else "CONT" if ro.eta2_content > ro.eta2_register * 2 else "MIX"
    flag_c = "REG" if rc.eta2_register > rc.eta2_content * 2 else "CONT" if rc.eta2_content > rc.eta2_register * 2 else "MIX"
    print(f"  {ro.model:<20} {ro.eta2_content:>8.3f} {ro.eta2_register:>8.3f} {flag_o:>8} │ "
          f"{rc.eta2_content:>8.3f} {rc.eta2_register:>8.3f} {flag_c:>8}")

print(f"\n{'MEAN':<22} {anova_orig['eta2_content'].mean():>8.3f} {anova_orig['eta2_register'].mean():>8.3f} "
      f"{'':>8} │ {anova_content['eta2_content'].mean():>8.3f} {anova_content['eta2_register'].mean():>8.3f}")

print(f"\nInterpretation:")
if anova_content['eta2_content'].mean() > anova_content['eta2_register'].mean() * 1.5:
    print("  R_HAT_content is CONTENT-DOMINANT ✓")
    print("  → The register-controlled direction captures semantic harmfulness")
elif anova_content['eta2_content'].mean() > anova_content['eta2_register'].mean():
    print("  R_HAT_content shows content > register (moderate)")
    print("  → Partial success: content signal is stronger but register remains")
else:
    print("  R_HAT_content is still register-dominant or mixed")
    print("  → The content contrast doesn't isolate a pure harmfulness direction")

In [ ]:
# ── CELL 5 — JBB CATEGORY CORRELATION ─────────────────────────────────────────

tensions_df = pd.read_csv(OUT_DIR / "jbb_tensions_content.csv")
meta_df     = pd.read_csv(OUT_DIR / "content_direction_results.csv")

cat_rank = {
    'Physical harm': 10, 'Economic harm': 9, 'Malware/Hacking': 8,
    'Government decision-making': 7, 'Sexual/Adult content': 6,
    'Fraud/Deception': 5, 'Harassment/Discrimination': 4,
    'Expert advice': 3, 'Privacy': 2, 'Disinformation': 1,
}

print("="*70)
print("R_HAT_CONTENT — JBB CATEGORY CORRELATION")
print("="*70)

for model_name in sorted(tensions_df['model'].unique()):
    sub = tensions_df[tensions_df['model']==model_name].sort_values('track')
    t = sub['tension'].values
    t_norm = (t - t.mean()) / t.std()
    cat_scores = np.array([cat_rank[c] for c in sub['category'].values])
    r, p = spearmanr(t_norm, cat_scores)
    print(f"  {model_name:<22} ρ(tension, cat_rank) = {r:+.3f}  p = {p:.4f}")

# Grand category means
print(f"\nGrand mean tension by category (across all models):")
all_cat_means = {}
for cat in cat_rank:
    vals = []
    for model_name in tensions_df['model'].unique():
        sub = tensions_df[(tensions_df['model']==model_name) & (tensions_df['category']==cat)]
        t = sub['tension'].values
        t_norm = (t - t.mean()) / t.std() if t.std() > 0 else t
        vals.append(t_norm.mean())
    all_cat_means[cat] = np.mean(vals)

for cat in sorted(all_cat_means, key=all_cat_means.get, reverse=True):
    print(f"  {cat:<30} τ̄ = {all_cat_means[cat]:>+.3f}  (rank {cat_rank[cat]})")

# Compare with original R_HAT category ordering
print(f"\ncos(R_HAT_content, R_HAT_orig) per model:")
for _, r in meta_df.iterrows():
    print(f"  {r['model']:<22} cos = {r['cos_with_orig']:>+.3f}")

In [ ]:
# ── CELL 6 — PAPER OUTPUT ─────────────────────────────────────────────────────

meta_df   = pd.read_csv(OUT_DIR / "content_direction_results.csv")
anova_c   = pd.read_csv(OUT_DIR / "anova_content_direction.csv")
anova_o   = pd.read_csv("./results/out_register/anova_results.csv")
tens_df   = pd.read_csv(OUT_DIR / "jbb_tensions_content.csv")

# Key stats
eta2_c_new = anova_c['eta2_content'].mean()
eta2_r_new = anova_c['eta2_register'].mean()
eta2_c_old = anova_o['eta2_content'].mean()
eta2_r_old = anova_o['eta2_register'].mean()
cos_mean   = meta_df['cos_with_orig'].abs().mean()

models = sorted(tens_df['model'].unique())
n_m = len(models)
T = np.zeros((n_m, 100))
for i, m in enumerate(models):
    sub = tens_df[tens_df['model']==m].sort_values('track')
    t = sub['tension'].values
    T[i] = (t - t.mean()) / t.std()

loo_aucs = []
for i in range(n_m):
    consensus = np.mean([T[j] for j in range(n_m) if j != i], axis=0)
    si = np.argsort(consensus)[::-1]
    labels = np.array([1]*10 + [0]*10)
    scores = np.array([T[i][j] for j in list(si[:10]) + list(si[-10:])])
    loo_aucs.append(roc_auc_score(labels, scores))

print("="*70)
print("FINAL COMPARISON")
print("="*70)
print(f"\n{'Metric':<40} {'R_HAT_orig':>12} {'R_HAT_content':>14}")
print("-"*68)
print(f"  {'η²_content (should be HIGH)' :<38} {eta2_c_old:>12.3f} {eta2_c_new:>14.3f}")
print(f"  {'η²_register (should be LOW)' :<38} {eta2_r_old:>12.3f} {eta2_r_new:>14.3f}")
print(f"  {'LOO-AUC₁₀':<38} {'0.963':>12} {np.mean(loo_aucs):>14.3f}")
print(f"  {'|cos(direction, R_HAT_orig)|':<38} {'1.000':>12} {cos_mean:>14.3f}")

print(f"\n\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Files in {OUT_DIR}:")
for f in sorted(OUT_DIR.glob("*")):
    print(f"  {f.name}")